# Real F1 radio dataset (OpenF1 + Whisper)

Builds the training data for the custom race-engineer model: every published team radio clip from 2023–2025 races, turned into text, with the engineer's line paired to the race situation at that moment (lap, position, gaps and trend, tyre and age, stops, rivals who just pitted, flags, rain).

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then run the cells top to bottom.

Everything is saved to your Google Drive (`MyDrive/f1-engineer/openf1`), and every step skips work already done, so if Colab disconnects just run the cells again and it carries on.

Rough times on the free tier (estimates): timing data ~30 min (OpenF1 allows 30 requests a minute), audio ~20–40 min, transcription ~30–60 min on the T4.

The radio audio belongs to Formula 1: keep the dataset private and don't publish it.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/f1-engineer/openf1'
YEARS = '2023 2024 2025'

## 1. Get the code

If the repo is private, GitHub will ask for a login: use your username and a [personal access token](https://github.com/settings/tokens) (read-only, repo scope) as the password. Or upload `tools/openf1_radio.py` with the folder icon on the left and skip the clone.

In [ ]:
!git clone --depth 1 https://github.com/Shryder25/F1-21_Race_Engineer.git /content/engineer 2>/dev/null || git -C /content/engineer pull
!pip install -q faster-whisper

## 2. Download races, timing data and radio clips

In [ ]:
!python /content/engineer/tools/openf1_radio.py fetch --years $YEARS --out "$OUT"

## 3. Transcribe the clips (GPU)

`small.en` is a good balance. `medium.en` is more accurate on noisy radio and about twice as slow.

In [ ]:
!python /content/engineer/tools/openf1_radio.py transcribe --out "$OUT" --model small.en --device cuda

## 4. Pair engineer lines with the race situation

The 3 most recent races are held out as the test set and never used for training.

In [ ]:
!python /content/engineer/tools/openf1_radio.py build --out "$OUT" --holdout 3

## 5. Look at the result

In [ ]:
import pandas as pd

train = pd.read_json(f'{OUT}/examples_train.jsonl', lines=True)
test = pd.read_json(f'{OUT}/examples_test.jsonl', lines=True)
print(len(train), 'training examples,', len(test), 'test examples')
print(train.explode('tags')['tags'].value_counts(dropna=False))
pd.set_option('display.max_colwidth', 200)
train.sample(min(15, len(train)), random_state=1)[
    ['circuit', 'driver', 'lap', 'situation', 'engineer', 'speaker_score']
]

Lines with a low `speaker_score` (under ~0.65) may be the driver talking, not the engineer. Skim a few; if many are wrong, tell Claude and it will tighten the filter.

When you're happy, the two `.jsonl` files in your Drive folder are the dataset for the fine-tune step.